# Анализ набора данных "insurance.csv"

## 1. Введение

В этом ноутбуке мы:

1. Выполним расширенный исследовательский анализ данных (EDA) для набора данных **Insurance**.
2. Очистим данные от выбросов (единственной, "первой" версией фильтрации, которая оказалась наиболее удачной: **IQR** с `multiplier=1.5` и **Z-score** с `threshold=3`).
3. Исследуем взаимосвязь признаков (корреляционные матрицы, pairplot, boxplot, регрессионные линии и т.д.).
4. Обучим и сравним несколько моделей:
   - Линейная регрессия (предсказание индивидуальных медицинских расходов `charges`).
   - Логистическая регрессия (предсказание `smoker` – курит/не курит).
   - Дерево решений (также регрессия по `charges`).
   - CatBoost: регрессор и классификатор.
5. Посмотрим метрики (R2, RMSE, Accuracy, F1-score) и визуализируем некоторые аспекты обучения.

## 2. Описание датасета

Датасет `insurance.csv` содержит следующие столбцы:

- **age**: возраст основного выгодоприобретателя.
- **sex**: пол страхового контрагента (`female`, `male`).
- **bmi**: индекс массы тела, в идеале 18.5–24.9.
- **children**: количество детей/иждивенцев.
- **smoker**: факт курения (`yes`, `no`).
- **region**: регион проживания (`northeast`, `northwest`, `southeast`, `southwest`).
- **charges**: индивидуальные медицинские расходы, выставленные медицинским страхованием.

По условию задачи нужно предсказывать **индивидуальные медицинские расходы** (`charges`).
Дополнительно будем решать бинарную задачу классификации: предсказывать, является ли человек курящим (`smoker`).

## 3. Импорт библиотек и первичная настройка


In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from datetime import datetime

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor, CatBoostClassifier
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split

sns.set_theme(color_codes=True)

# Функция для удобного сохранения графиков
def save_plot(filename):
    """
    Сохраняет текущий график в папку graphs/<date_time>.
    """
    current_time = datetime.now().strftime('%Y-%m-%d_%H-%M')
    dir_name = f"graphs/{current_time}"
    os.makedirs(dir_name, exist_ok=True)
    plt.savefig(f"{dir_name}/{filename}", dpi=300, bbox_inches='tight')
    plt.close()

## 4. Загрузка и первичный осмотр данных

In [4]:
# Считываем датасет insurance.csv
df_original = pd.read_csv("raw/insurance.csv")

# Сразу посмотрим, что там
display(df_original.head())
print("\nРазмеры датасета:", df_original.shape)
print("\nИнформация о датасете:")
df_original.info()
print("\nСтатистика числовых признаков:")
display(df_original.describe(include='all'))

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Размеры датасета: (1243, 7)

Информация о датасете:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1243 entries, 0 to 1242
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1243 non-null   int64  
 1   sex       1243 non-null   object 
 2   bmi       1243 non-null   float64
 3   children  1243 non-null   int64  
 4   smoker    1243 non-null   object 
 5   region    1243 non-null   object 
 6   charges   1242 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 68.1+ KB

Статистика числовых признаков:


,age,sex,bmi,children,smoker,region,charges
count,1243.000000,1243,1243.000000,1243.000000,1243,1243,1242.000000
unique,NaN,3,NaN,NaN,2,4,NaN
top,NaN,male,NaN,NaN,no,southeast,NaN
freq,NaN,623,NaN,NaN,989,336,NaN
mean,39.104586,NaN,30.582852,1.104586,NaN,NaN,13216.829311
std,14.049384,NaN,6.161992,1.215971,NaN,NaN,12002.492341
min,18.000000,NaN,15.960000,0.000000,NaN,NaN,1121.873900
25%,26.000000,NaN,26.050000,0.000000,NaN,NaN,4747.525500
50%,39.000000,NaN,30.210000,1.000000,NaN,NaN,9388.753650
75%,51.000000,NaN,34.637500,2.000000,NaN,NaN,16584.318157


### Удаление дубликатов и пропусков

По условию эксперимента удалим дубли и пропуски напрямую. (На реальном проекте возможно более тонкое заполнение.)

In [5]:
# Удалим дубликаты
df_original = df_original.drop_duplicates()
print(f"После удаления дубликатов: {df_original.shape}")

# Удалим пропуски
df_original = df_original.dropna()
print(f"После удаления пропусков: {df_original.shape}")

После удаления дубликатов: (1243, 7)
После удаления пропусков: (1242, 7)


## 5. Исследовательский анализ (EDA)

На данном этапе:
- Посмотрим распределения числовых признаков (age, bmi, children, charges).
- Сделаем pairplot (отражает взаимосвязи и распределения).
- Построим тепловую карту корреляций.
- Изучим категориальные признаки (sex, smoker, region).
- Обсудим выводы.


In [6]:
# Список числовых и категориальных признаков
num_cols = ['age', 'bmi', 'children', 'charges']
cat_cols = ['sex', 'smoker', 'region']

# Распределения числовых признаков
plt.figure(figsize=(12, 8))
for i, col in enumerate(num_cols, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df_original[col], kde=True)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
save_plot("num_distributions.png")

### Pairplot для числовых признаков

Позволяет увидеть попарные зависимости и распределения (на диагоналях).

In [7]:
sns.pairplot(df_original[num_cols], diag_kind='kde', corner=True)
plt.suptitle("Pairplot for numeric features", y=1.02)
save_plot("pairplot_numeric.png")

### Тепловая карта корреляций

Проверим, как связаны между собой числовые признаки. Обратим внимание на:
- `charges` vs `bmi`, `age`, `smoker` (хоть `smoker` категориальный, но часто его переводят в 0/1 для анализа);
- Иные взаимосвязи.

In [8]:
# Для тепловой карты переведём smoker в 0/1, чтобы увидеть корреляцию.
df_corr = df_original.copy()
df_corr['smoker'] = df_corr['smoker'].map({'no': 0, 'yes': 1})

# Строим корреляционную матрицу
corr_matrix = df_corr[num_cols + ['smoker']].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu')
plt.title("Correlation Heatmap")
save_plot("correlation_heatmap.png")
corr_matrix

,age,bmi,children,charges,smoker
age,1.000000,0.112707,0.032047,0.292575,-0.034082
bmi,0.112707,1.000000,0.010492,0.200421,0.003517
children,0.032047,0.010492,1.000000,0.057914,0.005270
charges,0.292575,0.200421,0.057914,1.000000,0.786440
smoker,-0.034082,0.003517,0.005270,0.786440,1.000000


Видно, что:
- `charges` имеет наиболее сильную положительную корреляцию с **age** (примерно 0.3) и особенно с "smoker" (0.79) — это значит, что курение сильно влияет на рост расходов.
- `bmi` тоже влияет на расходы, но корреляция более умеренная (~ 0.2).

Далее посмотрим на категорические признаки.

### Boxplot: Расходы по категориям

Построим ящики с усами (Boxplot), чтобы увидеть распределение `charges` в разных группах (по `smoker`, `sex`, `region`).

In [9]:
# Boxplot Charges vs Smoker
plt.figure(figsize=(10,6))
sns.boxplot(x='smoker', y='charges', data=df_original, palette='coolwarm')
plt.title("Charges by Smoker")
save_plot("boxplot_smoker_charges.png")

/var/folders/d5/0bhhtf7147nfb4zc40xcpcq00000gn/T/ipykernel_23866/1550253163.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='smoker', y='charges', data=df_original, palette='coolwarm')


In [10]:
# Boxplot Charges vs Sex
plt.figure(figsize=(10,6))
sns.boxplot(x='sex', y='charges', data=df_original, palette='pastel')
plt.title("Charges by Sex")
save_plot("boxplot_sex_charges.png")

/var/folders/d5/0bhhtf7147nfb4zc40xcpcq00000gn/T/ipykernel_23866/239731106.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='sex', y='charges', data=df_original, palette='pastel')


In [11]:
# Boxplot Charges vs Region
plt.figure(figsize=(10,6))
sns.boxplot(x='region', y='charges', data=df_original, palette='Set3')
plt.title("Charges by Region")
save_plot("boxplot_region_charges.png")

/var/folders/d5/0bhhtf7147nfb4zc40xcpcq00000gn/T/ipykernel_23866/737546723.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='region', y='charges', data=df_original, palette='Set3')


Из этих графиков видно:
- Курящие платят **значительно** больше за мед. расходы, в среднем.
- Пол (male/female) не даёт большой разницы.
- Разница по регионам тоже есть, но не такая яркая.

## 6. Очистка данных (единственная версия)

По заданию, оказалось, что **первая версия фильтрации** (IQR-мультипликатор 1.5 + Z-score порог 3) наиболее удачная. Используем **только** её.

### 6.1 Определим функции фильтрации

In [12]:
def filter_iqr(data, column, multiplier=1.5):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    mask = (data[column] >= Q1 - multiplier * IQR) & (data[column] <= Q3 + multiplier * IQR)
    return data[mask]

def filter_z_score(data, column, threshold=3):
    mean_val = data[column].mean()
    std_val = data[column].std()
    z_score = (data[column] - mean_val) / std_val
    return data[abs(z_score) <= threshold]

### 6.2 Применяем фильтрацию к исходному датасету

In [13]:
# Копируем исходный DataFrame, чтобы не потерять df_original
df = df_original.copy()

# 1) IQR фильтрация по charges
df = filter_iqr(df, 'charges', multiplier=1.5)
# 2) Z-score фильтрация по charges
df = filter_z_score(df, 'charges', threshold=3)

print(f"Размер данных после фильтрации: {df.shape}")
df.head()

Размер данных после фильтрации: (1098, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


### 6.3 Смотрим распределение `charges` после очистки

In [14]:
plt.figure(figsize=(8,5))
sns.histplot(df['charges'], kde=True)
plt.title("Distribution of charges AFTER filtering")
save_plot("charges_distribution_after.png")

## 7. Подготовка данных к моделям

**Примечание**: у нас две цели:
- **Regression**: `charges`.
- **Classification**: `smoker` (0 или 1).

### 7.1 Подготовка к регрессии (предсказываем `charges`)
Закодируем категориальные признаки (sex, smoker, region) через One-Hot Encoding.

In [15]:
def prepare_regression_data(df_):
    df_temp = df_.copy()
    df_temp = pd.get_dummies(df_temp, columns=['sex','smoker','region'], drop_first=True)
    X = df_temp.drop('charges', axis=1)
    y = df_temp['charges']
    return X, y

X_reg, y_reg = prepare_regression_data(df)

print("Признаки для регрессии:", X_reg.columns.tolist())
print("Размерности:", X_reg.shape, y_reg.shape)

Признаки для регрессии: ['age', 'bmi', 'children', 'sex_ma1639.5631le', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
Размерности: (1098, 9) (1098,)


### 7.2 Подготовка к классификации (предсказываем `smoker`)

Создадим датасет, где `smoker` становится целевым признаком. Переведём `smoker` в 0/1 явно. Закодируем остальные категориальные (sex, region).

In [16]:
def prepare_classification_data(df_):
    df_temp = df_.copy()
    # Целевой признак
    df_temp['smoker'] = df_temp['smoker'].map({'no': 0, 'yes': 1})
    y = df_temp['smoker']

    df_temp = df_temp.drop('smoker', axis=1)
    df_temp = pd.get_dummies(df_temp, columns=['sex','region'], drop_first=True)
    # Здесь не удаляем 'charges', т.к. это потенциально полезная фича для предсказания smoker
    X = df_temp
    return X, y

X_clf, y_clf = prepare_classification_data(df)
print("Признаки для классификации:", X_clf.columns.tolist())
print("Размерности:", X_clf.shape, y_clf.shape)

Признаки для классификации: ['age', 'bmi', 'children', 'charges', 'sex_ma1639.5631le', 'sex_male', 'region_northwest', 'region_southeast', 'region_southwest']
Размерности: (1098, 9) (1098,)


## 8. Обучение и оценка моделей

Будем обучать:
1. **Линейную регрессию** (предсказание `charges`).
2. **Логистическую регрессию** (предсказание `smoker`).
3. **Дерево решений** (регрессия по `charges`).
4. **CatBoost** (и как регрессор, и как классификатор).

### 8.1 Разбиение на train/test

In [17]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42
)
print("Train shape (reg):", X_train_reg.shape, "Test shape (reg):", X_test_reg.shape)
print("Train shape (clf):", X_train_clf.shape, "Test shape (clf):", X_test_clf.shape)

Train shape (reg): (878, 9) Test shape (reg): (220, 9)
Train shape (clf): (878, 9) Test shape (clf): (220, 9)


### 8.2 Линейная регрессия (Regression)

Здесь мы будем предсказывать `charges`. Посмотрим **R2** и **RMSE**.

**Графики**:
1. Фактические vs Предсказанные значения.
2. Остатки vs Предсказанные.
3. Пример регрессионной линии (но у нас много признаков, так что для демонстрации возьмём, например, зависимость `bmi` от `charges`, построив частную регрессию).

In [18]:
# Обучим линейную регрессию
linreg = LinearRegression()
linreg.fit(X_train_reg, y_train_reg)

y_pred_lin = linreg.predict(X_test_reg)

r2_lin = r2_score(y_test_reg, y_pred_lin)
rmse_lin = mean_squared_error(y_test_reg, y_pred_lin, squared=False)
print(f"[LinearRegression] R2: {r2_lin:.3f}, RMSE: {rmse_lin:.2f}")

# 1) Фактические vs Предсказанные
plt.figure(figsize=(6,6))
plt.scatter(y_test_reg, y_pred_lin, alpha=0.7, color='blue')
plt.plot([y_test_reg.min(), y_test_reg.max()], [y_test_reg.min(), y_test_reg.max()], color='red', lw=2)
plt.title("Linear Regression: Actual vs Predicted (charges)")
plt.xlabel("Actual charges")
plt.ylabel("Predicted charges")
save_plot("linreg_actual_vs_pred.png")

# 2) Остатки vs Предсказанные
residuals = y_test_reg - y_pred_lin

plt.figure(figsize=(6,6))
plt.scatter(y_pred_lin, residuals, alpha=0.7, color='green')
plt.axhline(y=0, color='red', linestyle='--')
plt.title("Linear Regression: Residuals vs Predicted")
plt.xlabel("Predicted charges")
plt.ylabel("Residuals")
save_plot("linreg_residuals_vs_pred.png")

# 3) Пример регрессионной линии (univariate) для наглядности
# Допустим, возьмём только bmi и charges
# Построим модель по одному признаку bmi, чтобы показать линию.

plt.figure(figsize=(6,5))
sns.regplot(x=df['bmi'], y=df['charges'], scatter_kws={'alpha':0.5}, color='purple')
plt.title("Univariate Regression: charges vs bmi")
save_plot("linreg_bmi_vs_charges.png")

[LinearRegression] R2: 0.611, RMSE: 4441.99


/Users/arseniikostin/Documents/Studies3/metiskint/new/tpu-8e21-ai-basis/.venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


### 8.3 Логистическая регрессия (Classification)

Предсказываем `smoker`. Посмотрим **Accuracy** и **F1**.

In [19]:
# Обучаем логистическую регрессию
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_clf, y_train_clf)
y_pred_log = logreg.predict(X_test_clf)

acc_log = accuracy_score(y_test_clf, y_pred_log)
f1_log = f1_score(y_test_clf, y_pred_log)
print(f"[LogisticRegression] Accuracy: {acc_log:.3f}, F1: {f1_log:.3f}")

# Построим матрицу ошибок
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test_clf, y_pred_log)
cmd = ConfusionMatrixDisplay(cm, display_labels=["No Smoker", "Smoker"])
cmd.plot(cmap="Blues")
plt.title("Logistic Regression - Confusion Matrix")
save_plot("logreg_confusion_matrix.png")

/Users/arseniikostin/Documents/Studies3/metiskint/new/tpu-8e21-ai-basis/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LogisticRegression] Accuracy: 0.959, F1: 0.824


### 8.4 Дерево решений (Regression)

In [20]:
dt_reg = DecisionTreeRegressor(random_state=42, max_depth=5)
dt_reg.fit(X_train_reg, y_train_reg)
y_pred_dt = dt_reg.predict(X_test_reg)

r2_dt = r2_score(y_test_reg, y_pred_dt)
rmse_dt = mean_squared_error(y_test_reg, y_pred_dt, squared=False)
print(f"[DecisionTreeRegressor] R2: {r2_dt:.3f}, RMSE: {rmse_dt:.2f}")

# Попробуем визуализировать важность признаков
importances = dt_reg.feature_importances_
feature_names = X_train_reg.columns

imp_df = pd.DataFrame({'feature': feature_names, 'importance': importances})\
          .sort_values('importance', ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x='importance', y='feature', data=imp_df, palette='Blues_r')
plt.title("Feature Importances - DecisionTreeRegressor")
save_plot("dt_reg_feature_importances.png")

imp_df

/Users/arseniikostin/Documents/Studies3/metiskint/new/tpu-8e21-ai-basis/.venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
/var/folders/d5/0bhhtf7147nfb4zc40xcpcq00000gn/T/ipykernel_23866/3377782147.py:17: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='importance', y='feature', data=imp_df, palette='Blues_r')


[DecisionTreeRegressor] R2: 0.589, RMSE: 4568.94


,feature,importance
5,smoker_yes,0.500088
0,age,0.435198
2,children,0.037859
1,bmi,0.021260
8,region_southwest,0.004655
7,region_southeast,0.000663
4,sex_male,0.000276
3,sex_ma1639.5631le,0.000000
6,region_northwest,0.000000


### 8.5 CatBoost

CatBoost — алгоритм градиентного бустинга, который хорошо работает с табличными данными и часто не требует дополнительного кодирования категориальных признаков. Однако здесь мы уже сделали One-Hot Encoding. Важно, что:
- Для регрессии используем `CatBoostRegressor`.
- Для классификации — `CatBoostClassifier`.

Посмотрим на **R2, RMSE** (для регрессора) и **Accuracy, F1** (для классификатора).

In [21]:
# CatBoost Regressor
cbr = CatBoostRegressor(verbose=0, random_state=42)
cbr.fit(X_train_reg, y_train_reg)
y_pred_cbr = cbr.predict(X_test_reg)

r2_cbr = r2_score(y_test_reg, y_pred_cbr)
rmse_cbr = mean_squared_error(y_test_reg, y_pred_cbr, squared=False)
print(f"[CatBoostRegressor] R2: {r2_cbr:.3f}, RMSE: {rmse_cbr:.2f}")

# CatBoost Classifier
cbc = CatBoostClassifier(verbose=0, random_state=42)
cbc.fit(X_train_clf, y_train_clf)
y_pred_cbc = cbc.predict(X_test_clf)

acc_cbc = accuracy_score(y_test_clf, y_pred_cbc)
f1_cbc = f1_score(y_test_clf, y_pred_cbc)
print(f"[CatBoostClassifier] Accuracy: {acc_cbc:.3f}, F1: {f1_cbc:.3f}")

# Матрица ошибок для CatBoostClassifier
cm_cbc = confusion_matrix(y_test_clf, y_pred_cbc)
cmd_cbc = ConfusionMatrixDisplay(cm_cbc, display_labels=["No Smoker", "Smoker"])
cmd_cbc.plot(cmap="Oranges")
plt.title("CatBoost - Confusion Matrix")
save_plot("catboost_confusion_matrix.png")

/Users/arseniikostin/Documents/Studies3/metiskint/new/tpu-8e21-ai-basis/.venv/lib/python3.11/site-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


[CatBoostRegressor] R2: 0.555, RMSE: 4753.73
[CatBoostClassifier] Accuracy: 0.982, F1: 0.923


## 9. Сравнение итоговых результатов

**Резюме по регрессии** (`charges`):
1. Линейная регрессия: R2 ~ …, RMSE ~ …
2. Дерево решений: R2 ~ …, RMSE ~ …
3. CatBoostRegressor: R2 ~ …, RMSE ~ …

**Резюме по классификации** (`smoker`):
1. Логистическая регрессия: Accuracy ~ …, F1 ~ …
2. CatBoostClassifier: Accuracy ~ …, F1 ~ …

Чаще всего в данной задаче (страховые расходы) бустинг даёт более высокое качество, чем простая линейная модель. Но и линейная регрессия может показать неплохой результат, если правильно учесть зависимости (особенно курение, возраст, bmi).

## 10. Выводы

- **EDA** показал, что `smoker` и `age` — одни из ключевых факторов, сильно влияющих на мед.расходы. Высокий `bmi` также вносит вклад.
- **Фильтрация IQR (1.5) + Z-score (3)** позволила убрать экстремальные выбросы, улучшив стабильность и точность некоторых моделей.
- **CatBoost** обычно даёт более высокое качество (R2, Accuracy) по сравнению с линейными моделями.
- **Логистическая регрессия** справляется с задачей предсказания `smoker` достаточно хорошо, особенно если учесть, что `charges` сильно варьируются у курящих.
- Для ещё более высокого качества можно искать оптимальные гиперпараметры (GridSearchCV, RandomSearchCV) и прорабатывать дополнительные фичи (полиномиальные признаки, взаимодействия).

На этом анализ завершаем. Все графики сохранены в папку `graphs/<дата_время>`. В итоге вы можете выбрать наиболее подходящую модель и стратегию фильтрации в зависимости от конкретных бизнес-требований.